# 지하철 30분 데이터 월별 집계 파이프라인

- 실컬럼명 기준: `YEAR, MONTH, DAY, HOUR, HALF_HOUR, STATION_ID, LINE_NM, STATION_NM, GETON_CNT, GETOFF_CNT`
- 구분자: `|`
- 폴더 구조: `연/월/(월)/일별 csv` 모두 대응
- 결과: 월별 집계 CSV + ZIP 생성

In [ ]:
import argparse
import sys

from pathlib import Path
from datetime import datetime
import zipfile

import pandas as pd


REQUIRED_OUTPUT_COLUMNS = [
    # [수정] 지하철 데이터는 호선명(LINE_NM)과 역명(STATION_NM)을 함께 사용합니다.
    "YEAR",
    "MONTH",
    "DAY",
    "HOUR",
    "HALF_HOUR",
    "STATION_ID",
    "LINE_NM",
    "STATION_NM",
    "GETON_CNT",
    "GETOFF_CNT",
]

# Allow common source-column variants but force final schema to requested names.
COLUMN_SYNONYMS = {
    # [수정] 지하철 원본의 한글 컬럼명을 표준 영문 컬럼명으로 맞춥니다.
    "YEAR": ["YEAR", "년(YEAR)"],
    "MONTH": ["MONTH", "월(MONTH)"],
    "DAY": ["DAY", "일(DAY)"],
    "HOUR": ["HOUR", "시간(HOUR)"],
    "HALF_HOUR": ["HALF_HOUR", "분_30분단위(HALF_HOUR)"],
    "STATION_ID": ["STATION_ID", "역ID(STATION_ID)", "역_ID(STATION_ID)"],
    "LINE_NM": ["LINE_NM", "호선명(LINE_NM)", "노선명(LINE_NM)", "호선(LINE_NM)"],
    "STATION_NM": ["STATION_NM", "역명(STATION_NM)", "STATION,_NM"],
    "GETON_CNT": ["GETON_CNT", "승차인원(GETON_CNT)", "승차총승객수(GETON_CNT)", "승하차인원(GETON_CNT)"],
    "GETOFF_CNT": ["GETOFF_CNT", "하차인원(GETOFF_CNT)", "하차총승객수(GETOFF_CNT)"],
}


def normalize_col_name(col: str) -> str:
    return str(col).strip().replace("\ufeff", "")


def harmonize_columns(df: pd.DataFrame) -> pd.DataFrame:
    src_cols = {normalize_col_name(c): c for c in df.columns}
    rename_map = {}

    for target, candidates in COLUMN_SYNONYMS.items():
        found = None
        for c in candidates:
            c_norm = normalize_col_name(c)
            if c_norm in src_cols:
                found = src_cols[c_norm]
                break
        if found is not None:
            rename_map[found] = target

    df = df.rename(columns=rename_map)

    missing = [c for c in REQUIRED_OUTPUT_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    return df[REQUIRED_OUTPUT_COLUMNS].copy()


def read_daily_csv(file_path: Path, sep: str) -> pd.DataFrame:
    # [수정] 지하철 원본 CSV는 환경에 따라 utf-8-sig/cp949/euc-kr 인코딩이 섞일 수 있어 순차 시도합니다.
    last_error = None
    for encoding in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(file_path, sep=sep, engine="python", dtype=str, encoding=encoding)
            break
        except UnicodeDecodeError as e:
            last_error = e
    else:
        raise last_error
    df = harmonize_columns(df)

    for c in ["GETON_CNT", "GETOFF_CNT"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    return df


def aggregate_monthly(month_df: pd.DataFrame) -> pd.DataFrame:
    # [수정] 버스 정류소 기준이 아니라 지하철 호선/역 + 30분 단위 기준으로 월별 합산합니다. DAY는 월집계라 제거합니다.
    group_cols = ["YEAR", "MONTH", "HOUR", "HALF_HOUR", "STATION_ID", "LINE_NM", "STATION_NM"]

    out = (
        month_df.groupby(group_cols, dropna=False, as_index=False)[["GETON_CNT", "GETOFF_CNT"]]
        .sum()
        .sort_values(group_cols)
    )
    return out


def zip_output_folder(output_dir: Path) -> Path:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = output_dir.parent / f"monthly_aggregates_{ts}.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in output_dir.glob("*.csv"):
            zf.write(file, arcname=file.name)

    return zip_path


def discover_month_folders(input_root: Path) -> list[Path]:
    # Case A: E:/2022/202201 ...
    direct_months = [
        p for p in input_root.iterdir() if p.is_dir() and len(p.name) == 6 and p.name.isdigit()
    ]
    if direct_months:
        return sorted(direct_months)

    # Case B: E:/2022/202201 ... where input_root is E:/
    year_dirs = [p for p in input_root.iterdir() if p.is_dir() and len(p.name) == 4 and p.name.isdigit()]
    nested_months = []
    for y in year_dirs:
        nested_months.extend(
            [p for p in y.iterdir() if p.is_dir() and len(p.name) == 6 and p.name.isdigit()]
        )
    return sorted(nested_months)


def is_under_path(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False


def discover_csv_files(input_root: Path, output_dir: Path) -> list[Path]:
    # [수정] 월 폴더명이 202201 형태가 아니어도 동작하도록 input_root 아래 CSV를 재귀 탐색합니다.
    if not input_root.exists():
        raise FileNotFoundError(f"Input root does not exist: {input_root}")

    csv_files = [
        p for p in input_root.rglob("*.csv")
        if p.is_file() and not is_under_path(p, output_dir)
    ]

    subway_files = [p for p in csv_files if "SUBWAY" in p.name.upper()]
    return sorted(subway_files or csv_files)


def run_pipeline(input_root: Path, output_dir: Path, sep: str) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    # [수정] 폴더명이 아니라 CSV 내부의 YEAR/MONTH 컬럼 기준으로 월별 집계합니다.
    daily_files = discover_csv_files(input_root, output_dir)
    if not daily_files:
        raise FileNotFoundError(
            f"No csv files found under: {input_root}. Check input_root path and extracted data folder."
        )

    print(f"[INFO] Found csv files: {len(daily_files):,}")
    monthly_frames_by_key = {}

    for f in daily_files:
        try:
            daily_df = read_daily_csv(f, sep=sep)
        except Exception as e:
            print(f"[WARN] Skip file due to parse/schema issue: {f} -> {e}")
            continue

        # [수정] 한 파일 안에 여러 월이 섞여도 YEAR/MONTH 기준으로 나누어 부분 집계합니다.
        for (yy, mm), part in daily_df.groupby(["YEAR", "MONTH"], dropna=False):
            key = (str(yy).zfill(4), str(mm).zfill(2))
            monthly_frames_by_key.setdefault(key, []).append(aggregate_monthly(part))

    written_files = []

    for (yy, mm), frames in sorted(monthly_frames_by_key.items()):
        month_df = pd.concat(frames, ignore_index=True)
        monthly_agg = aggregate_monthly(month_df)

        # [수정] 산출 파일명도 subway 기준으로 변경했습니다.
        out_file = output_dir / f"subway_30min_monthly_{yy}_{mm}.csv"
        monthly_agg.to_csv(out_file, sep=sep, index=False, encoding="utf-8-sig")
        written_files.append(out_file)
        print(f"[OK] {out_file} rows={len(monthly_agg):,}")

    if not written_files:
        raise RuntimeError("No monthly output was generated. Check folder structure/columns/separator.")

    zip_path = zip_output_folder(output_dir)
    print("\n=== Pipeline Completed ===")
    print(f"Monthly files: {len(written_files)}")
    print(f"Zip: {zip_path}")


def main() -> None:
    # [수정] CLI 설명도 지하철 데이터 파이프라인 기준으로 변경했습니다.
    parser = argparse.ArgumentParser(description="Subway 30-min daily CSV -> monthly aggregate CSV pipeline")
    parser.add_argument(
        "--input-root",
        required=True,
        help="Root folder (E drive) containing year/month/day CSV hierarchy, e.g. E:/subway_data",
    )
    parser.add_argument(
        "--output-dir",
        required=True,
        help="Output folder for monthly CSV files",
    )
    parser.add_argument(
        "--sep",
        default="|",
        help="CSV separator (default: |)",
    )

    args = parser.parse_args()

    run_pipeline(
        input_root=Path(args.input_root),
        output_dir=Path(args.output_dir),
        sep=args.sep,
    )


# [수정] 노트북 실행 전용: Jupyter/빅데이터캠퍼스에서는 argparse main()을 자동 실행하지 않습니다.
# 아래 실행 예시 셀에서 run_pipeline(input_root=..., output_dir=..., sep=...)만 직접 실행하세요.
# 터미널에서 .py 파일로 실행할 때만 아래 2줄의 주석을 해제하면 됩니다.
# if __name__ == "__main__":
#     main()



In [ ]:
# 실행 예시 (실서버 경로에 맞게 수정해서 실행)
input_root = Path(r"E:\2022")
output_dir = Path(r"E:\2022\_monthly_out")
sep = "|"

# [수정] 실행 전 입력 폴더와 CSV 탐색 결과를 먼저 확인합니다.
print("input_root exists:", input_root.exists())
print("sample csv files:", list(input_root.rglob("*.csv"))[:5])

run_pipeline(input_root=input_root, output_dir=output_dir, sep=sep)